# Plot data for Kolen-Pollack decay rate study in SSNs

In this script, we plot our study on the effect of the Kolen-Pollack weight decay rate on the training performance of spiking sampling networks (SSN). We scan through the weight decay rate in both nosie scenarios (**synpatic noise** and **plasticity noise**) and for two difference **noise levels** show that
1. the best decay rate depends on the noise levels and
2. weight decay cannot balance out the effect of plasticity noise, because the decay would have to be so strong that it would suppress all "functional" learning. 

In [ ]:
import math
from collections import defaultdict
from pathlib import Path
from typing import Optional

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
import yaml
from matplotlib.lines import Line2D

In [ ]:
mpl.style.use("../../mystyle.mpl")

In [ ]:
FIG_DIR = Path("../../figs/ssn/")
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("../../results/ssn")

## Loading helpers

In [ ]:
def load_kp_runs(
    results_dir: Path,
    noise_param: str | list[str],
    lr_param: str | list[str],
) -> dict:
    """Load KP-sweep runs, grouped by (noise_level, kp_lr).

    Returns:
        {noise_level: {"dkl":  {kp_lr: DataFrame},
                       "asym": {kp_lr: DataFrame},
                       "params": dict}}
    """
    if isinstance(noise_param, str):
        noise_param = [noise_param]
    if isinstance(lr_param, str):
        lr_param = [lr_param]

    store: dict = defaultdict(lambda: {"dkl": {}, "asym": {}, "params": None})
    params_cache: dict[str, dict] = {}

    for npz_path in sorted(results_dir.glob("*.npz")):
        stem = npz_path.stem
        sweep_id, seed_id = stem.split("_", 1)

        if sweep_id not in params_cache:
            with open(results_dir / f"{sweep_id}_params.yaml") as f:
                params_cache[sweep_id] = yaml.safe_load(f)
        p = params_cache[sweep_id]

        noise_val = round(float(_get_nested(p, noise_param)), 10)
        lr_val = round(float(_get_nested(p, lr_param)), 15)

        if store[noise_val]["params"] is None:
            store[noise_val]["params"] = p

        data = np.load(npz_path)
        store[noise_val]["dkl"].setdefault(lr_val, {})[stem] = data["dkls"]
        store[noise_val]["asym"].setdefault(lr_val, {})[stem] = data["all_asym"]

    return {
        noise: {
            "dkl": {lr: pd.DataFrame(v) for lr, v in sorted(d["dkl"].items())},
            "asym": {lr: pd.DataFrame(v) for lr, v in sorted(d["asym"].items())},
            "params": d["params"],
        }
        for noise, d in sorted(store.items())
    }

## Plotting helpers

In [ ]:
def add_stats(df, regex=r"^\d"):
    df["mean"] = df.filter(regex=regex).mean(axis=1)
    df["std"] = df.filter(regex=regex).std(axis=1)
    df["median"] = df.filter(regex=regex).quantile(q=0.5, axis=1)
    df["lower_q"] = df.filter(regex=regex).quantile(q=0.25, axis=1)
    df["upper_q"] = df.filter(regex=regex).quantile(q=0.75, axis=1)


def extract_data(df_dict):
    """Extract final-step median and IQR errors from a dict of DataFrames."""
    median = np.array([df["median"].iloc[-1] for df in df_dict.values()])
    upper_q = np.array([df["upper_q"].iloc[-1] for df in df_dict.values()])
    lower_q = np.array([df["lower_q"].iloc[-1] for df in df_dict.values()])
    u_err = median - lower_q
    l_err = upper_q - median
    return median, np.array([u_err, l_err])


def ebar(
    ax,
    dat: dict[float, npt.NDArray],
    pos: int,
    color: str,
    label: Optional[str] = None,
    **kwargs,
):
    u_err = dat["median"].iloc[-1] - dat["lower_q"].iloc[-1]
    l_err = dat["upper_q"].iloc[-1] - dat["median"].iloc[-1]
    ax.errorbar(
        [pos],
        [dat["median"].iloc[-1]],
        yerr=np.array([[u_err], [l_err]]),
        capsize=4,
        color=color,
        label=label,
        **kwargs,
    )
    return ax


def fill_between(ax, *args, **kwargs):
    kwargs.setdefault("linewidth", 0.0)
    return ax.fill_between(*args, **kwargs)

# Kolen-Pollack weight decay rate

## Synaptic noise

In [ ]:
noise_levels = [0.4, 0.8]
kp_lr_syn = 1e-6 * np.array([1, 2, 4, 8, 16, 32, 64, 120, 240, 480])
print(kp_lr_syn)

kp_data_syn = load_kp_runs(
    RESULTS_DIR / "syn_noise_kp_lr",
    noise_param="init_noise",
    lr_param=["lr", "kp"],
)

for noise in kp_data_syn.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

## Plasticity noise

In [ ]:
kp_lr_plast = np.array([1e-5 * 2**i for i in [-3, -2, -1, 0, 1, 2, 3, 4]])
print(kp_lr_plast)

kp_data_plast = load_kp_runs(
    RESULTS_DIR / "plast_noise_kp_lr",
    noise_param=["stdp", "ws", "noise"],
    lr_param=["lr", "kp"],
)

for noise in kp_data_plast.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

## Plot

In [ ]:
fig, ax = plt.subplots(
    2, 2, sharex="col", figsize=(18 / 2.54, 11 / 2.54), tight_layout=True
)

ax[1, 0].set_xscale("log")
ax[1, 0].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["dkl"]))),
}
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 4
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 5
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 0].get_xlim()
ax[1, 0].axhline(
    ref_syn_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_INIT_NOISE} = 0.4",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.4][1]] * 2,
    [ref_syn_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 0].axhline(
    ref_syn_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_INIT_NOISE} = 0.8",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.8][1]] * 2,
    [ref_syn_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

ax[0, 0].set_title("synaptic noise", pad=45)
ax[1, 0].set_xlim(xlims)
hs, ls = ax[0, 0].get_legend_handles_labels()
h, l = ax[1, 0].get_legend_handles_labels()
ax[0, 0].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)

ax[1, 0].set_xlabel("weight decay rate $\lambda$")
ax[1, 0].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 0].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

# Plasticity noise
ax[1, 1].set_xscale("log")
ax[1, 1].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["dkl"]))),
}
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 1
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 2
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 1].get_xlim()
ax[1, 1].axhline(
    ref_plast_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_PLAST_NOISE} = 0.4",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.4][1]] * 2,
    [ref_plast_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 1].axhline(
    ref_plast_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_PLAST_NOISE} = 0.8",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.8][1]] * 2,
    [ref_plast_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

hs, ls = ax[0, 1].get_legend_handles_labels()
h, l = ax[1, 1].get_legend_handles_labels()
ax[0, 1].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)
ax[0, 1].set_title("plasticity noise", pad=45)
ax[0, 1].set_xlim(xlims)

ax[1, 1].set_xlabel("weight decay rate $\lambda$")
ax[1, 1].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 1].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

### Load synaptic noise data

In [ ]:
noise_levels = [0.4, 0.8]
kp_lr_syn = 1e-6 * np.array([1, 2, 4, 8, 16, 32, 64, 120, 240, 480])
print(kp_lr_syn)

kp_data_syn = load_kp_runs(
    RESULTS_DIR / "syn_noise_kp_lr",
    noise_param="init_noise",
    lr_param=["lr", "kp"],
)

for noise in kp_data_syn.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

###  Load plasticity noise data

In [ ]:
kp_lr_plast = np.array([1e-5 * 2**i for i in [-3, -2, -1, 0, 1, 2, 3, 4]])
print(kp_lr_plast)

kp_data_plast = load_kp_runs(
    RESULTS_DIR / "plast_noise_kp_lr",
    noise_param=["stdp", "ws", "noise"],
    lr_param=["lr", "kp"],
)

for noise in kp_data_plast.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

### Plot

In [ ]:
fig, ax = plt.subplots(
    2, 2, sharex="col", figsize=(18 / 2.54, 11 / 2.54), tight_layout=True
)

ax[1, 0].set_xscale("log")
ax[1, 0].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["dkl"]))),
}
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 4
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 5
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 0].get_xlim()
ax[1, 0].axhline(
    ref_syn_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_INIT_NOISE} = 0.4",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.4][1]] * 2,
    [ref_syn_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 0].axhline(
    ref_syn_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_INIT_NOISE} = 0.8",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.8][1]] * 2,
    [ref_syn_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

ax[0, 0].set_title("synaptic noise", pad=45)
ax[1, 0].set_xlim(xlims)
hs, ls = ax[0, 0].get_legend_handles_labels()
h, l = ax[1, 0].get_legend_handles_labels()
ax[0, 0].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)

ax[1, 0].set_xlabel("weight decay rate $\lambda$")
ax[1, 0].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 0].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

# Plasticity noise
ax[1, 1].set_xscale("log")
ax[1, 1].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["dkl"]))),
}
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 1
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 2
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 1].get_xlim()
ax[1, 1].axhline(
    ref_plast_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_PLAST_NOISE} = 0.4",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.4][1]] * 2,
    [ref_plast_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 1].axhline(
    ref_plast_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_PLAST_NOISE} = 0.8",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.8][1]] * 2,
    [ref_plast_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

hs, ls = ax[0, 1].get_legend_handles_labels()
h, l = ax[1, 1].get_legend_handles_labels()
ax[0, 1].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)
ax[0, 1].set_title("plasticity noise", pad=45)
ax[0, 1].set_xlim(xlims)

ax[1, 1].set_xlabel("weight decay rate $\lambda$")
ax[1, 1].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 1].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

In [ ]:
fig.savefig(FIG_DIR / "kp_lr.png", bbox_inches="tight", dpi=300)
fig.savefig(FIG_DIR / "kp_lr.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "kp_lr.svg", bbox_inches="tight")